## Short-term memory
​
**Overview**

Memory is a system that remembers information about previous interactions. For AI agents, memory is crucial because it lets them remember previous interactions, learn from feedback, and adapt to user preferences. As agents tackle more complex tasks with numerous user interactions, this capability becomes essential for both efficiency and user satisfaction.
Short term memory lets your application remember previous interactions within a single thread or conversation.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint

In [3]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [4]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


In [6]:


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


In [7]:

agent = create_agent(
    model=model_advanced,
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)


In [8]:

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)


{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='86b52574-9b2b-4b5b-ba5c-27df8a8b6683'),
  AIMessage(content='Hi Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-5.6-luna-pro', 'id': 'gen-1789084412-2dBmFPV3B3twxK8OuqQd', 'created': 1789084412, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0004426, 'cost_details': {'upstream_inference_completions_cost': 0.0001296, 'upstream_inference_prompt_cost': 0.000313, 'upstream_inference_cost': 0.0004426}}, id='lc_run--01a08dbd-a116-7853-b49e-7b6d5e671aeb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1565, 'output_tokens': 108, 'total_tokens': 1673, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 34}})]}

In [9]:

agent.invoke({"messages": "write a short poem about cats"}, config)


{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='86b52574-9b2b-4b5b-ba5c-27df8a8b6683'),
  AIMessage(content='Hi Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-5.6-luna-pro', 'id': 'gen-1789084412-2dBmFPV3B3twxK8OuqQd', 'created': 1789084412, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0004426, 'cost_details': {'upstream_inference_completions_cost': 0.0001296, 'upstream_inference_prompt_cost': 0.000313, 'upstream_inference_cost': 0.0004426}}, id='lc_run--01a08dbd-a116-7853-b49e-7b6d5e671aeb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1565, 'output_tokens': 108, 'total_tokens': 1673, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 34}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={

In [10]:

agent.invoke({"messages": "now do the same but for dogs"}, config)


{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='86b52574-9b2b-4b5b-ba5c-27df8a8b6683'),
  AIMessage(content='Hi Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-5.6-luna-pro', 'id': 'gen-1789084412-2dBmFPV3B3twxK8OuqQd', 'created': 1789084412, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0004426, 'cost_details': {'upstream_inference_completions_cost': 0.0001296, 'upstream_inference_prompt_cost': 0.000313, 'upstream_inference_cost': 0.0004426}}, id='lc_run--01a08dbd-a116-7853-b49e-7b6d5e671aeb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1565, 'output_tokens': 108, 'total_tokens': 1673, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 34}}),
  HumanMessage(content='write a short poem about cats', additional_kwargs={

In [11]:

final_response = agent.invoke({"messages": "what's my name?"}, config)


In [12]:
final_response

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='86b52574-9b2b-4b5b-ba5c-27df8a8b6683'),
  AIMessage(content='Soft paws patter through the night,  \nWhiskers gleam in moonlit light.  \nWith a purr, they curl and nap—  \nTiny kings in velvet laps.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-5.6-luna-pro', 'id': 'gen-1789084434-Z2qeGhthQFxpmv5iMTgj', 'created': 1789084434, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0007224, 'cost_details': {'upstream_inference_completions_cost': 0.0003684, 'upstream_inference_prompt_cost': 0.000354, 'upstream_inference_cost': 0.0007224}}, id='lc_run--01a08dbd-fb82-71c2-933e-db395a3ac37e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1770, 'output_tokens': 307, 'total_tokens': 2077, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning

In [ ]:

final_response["messages"][-1].pretty_print()
"""
================================== Ai Message ==================================

Your name is Bob. You told me that earlier.
If you'd like me to call you a nickname or use a different name, just say the word.
"""